# Module 04 — Workflow: prompt chaining

**THE ONE IDEA:** a chain is LLM → **gate** → LLM → **gate** → LLM, and the gate is
ordinary Python you wrote. Control flow lives in your code, not in the model.

This is the first of Anthropic's five workflow patterns. Blocks B exists *before* the
agent blocks on purpose: once you have built a chain by hand, "most things called
agents are workflows" stops being a slogan and becomes something you have felt.

The gate is the whole point. It can **stop the chain**, and a model cannot overrule it.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client
from pydantic import BaseModel

client, MODEL, _ = get_client("openai")

APPLICATION = ("Applicant Sarah Chen, earns 74000 a year, existing debts 950 a month, "
               "wants to borrow 285000 on a property valued at 320000. First-time buyer.")

class Loan(BaseModel):
    applicant: str
    income: float
    monthly_debts: float
    amount: float
    property_value: float

def call(prompt, schema=None, max_tok=400):
    """One LLM step. With a schema it returns a typed object, else text."""
    kw = {}
    if schema:
        s = schema.model_json_schema(); s["additionalProperties"] = False
        kw["response_format"] = {"type": "json_schema",
                                 "json_schema": {"name": "out", "strict": True, "schema": s}}
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok,
                                       messages=[{"role": "user", "content": prompt}], **kw)
    txt = r.choices[0].message.content
    return schema.model_validate_json(txt) if schema else txt.strip()

print("chain ready")

## Step 1 → Gate 1

Extract, then check eligibility **in Python**. If LTV is over policy, the chain stops.
No later step gets a chance to talk us out of it.

In [ ]:
loan = call(f"Extract the loan details.\n\n{APPLICATION}", schema=Loan)
print("STEP 1 extracted:", loan.model_dump())

ltv = loan.amount / loan.property_value * 100
MAX_LTV = 95.0
print(f"\nGATE 1  LTV = {ltv:.1f}%  (policy max {MAX_LTV}%)")

if ltv > MAX_LTV:
    print("  -> REJECTED. Chain halts here. Steps 2 and 3 never run.")
    raise SystemExit
print("  -> pass")

## Step 2 → Gate 2

Affordability. Again the decision is arithmetic, not a model's opinion.

In [ ]:
monthly_income = loan.income / 12
ratio = loan.monthly_debts / monthly_income * 100
MAX_RATIO = 45.0

summary = call(f"In one sentence, summarise this applicant's financial position: "
               f"income {loan.income}, monthly debts {loan.monthly_debts}, "
               f"borrowing {loan.amount}.")
print("STEP 2 summary:", summary[:160])

print(f"\nGATE 2  debt-to-income = {ratio:.1f}%  (policy max {MAX_RATIO}%)")
passed = ratio <= MAX_RATIO
print("  ->", "pass" if passed else "REJECTED. Chain halts.")

## Step 3 — only reachable if both gates passed

In [ ]:
if passed:
    letter = call(f"Write a 2-sentence approval note for {loan.applicant}, "
                  f"borrowing {loan.amount:.0f} at {ltv:.1f}% LTV. Formal tone.")
    print("STEP 3 output:\n", letter)

## The lesson

In [ ]:
print("LESSON — control flow is IN YOUR CODE. Three LLM calls, two gates, and the")
print("gates are `if` statements over typed numbers.")
print()
print("  step 1  LLM   extract          -> Loan object")
print("  gate 1  YOU   ltv <= 95        -> can HALT")
print("  step 2  LLM   summarise")
print("  gate 2  YOU   dti <= 45        -> can HALT")
print("  step 3  LLM   write letter")
print()
print("The path is identical on every input. Cost and latency are predictable, and")
print("you can unit-test the gates without an API key. That is a WORKFLOW.")
print()
print("An AGENT (module 07 on) hands the sequencing to the model. You trade this")
print("predictability for the ability to handle inputs you could not enumerate.")
print("Most production systems should stop here.")

---

**Next:** `05_workflow_routing.ipynb`